In [9]:
import os
from google.colab import drive
drive.mount('/content/drive')

# KHAI BÁO ĐƯỜNG DẪN MODULE
PROJECT_PATH = '/content/drive/MyDrive/DeTaiHocSauKetHopDoAnTongHop/Stock_Forecasting_Project'
SOURCE_CODE_PATH = os.path.join(PROJECT_PATH, '04_Source_Code')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
%%writefile /content/drive/MyDrive/DeTaiHocSauKetHopDoAnTongHop/Stock_Forecasting_Project/04_Source_Code/data_ingestion.py
import pandas as pd
import numpy as np
import yfinance as yf
from typing import Optional

class DataIngestor:
    """Module chịu trách nhiệm thu thập và làm sạch dữ liệu chứng khoán."""
    def __init__(self, ticker: str, start_date: str, end_date: str):
        self.ticker = ticker
        self.start_date = start_date
        self.end_date = end_date

    def fetch_stock_data(self) -> pd.DataFrame:
        """Lấy dữ liệu OHLCV và xử lý dữ liệu trống."""
        print(f"Fetching data for {self.ticker}...")
        df = yf.download(self.ticker, start=self.start_date, end=self.end_date, auto_adjust=True)

        # Xử lý các ngày nghỉ lễ/cuối tuần bị thiếu bằng Forward Fill
        df = df.reindex(pd.date_range(start=self.start_date, end=self.end_date, freq='B'))
        df = df.ffill()

        # Xử lý MultiIndex nếu yfinance trả về
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)

        df.index.name = 'time'
        return df.dropna()

Overwriting /content/drive/MyDrive/DeTaiHocSauKetHopDoAnTongHop/Stock_Forecasting_Project/04_Source_Code/data_ingestion.py


In [ ]:
%%writefile /content/drive/MyDrive/DeTaiHocSauKetHopDoAnTongHop/Stock_Forecasting_Project/04_Source_Code/feature_engineering.py
import pandas as pd
import numpy as np
import pandas_ta as ta
from sklearn.preprocessing import StandardScaler
from typing import Tuple, List

class FeatureEngineer:
    """Tính toán Technical Indicators và chuẩn bị Sliding Window (Anti-Leakage)."""
    def __init__(self, window_size: int = 30):
        self.window_size = window_size
        self.scaler = StandardScaler()

    def add_indicators(self, df: pd.DataFrame) -> pd.DataFrame:
        """Tính toán RSI, MACD, Bollinger Bands."""
        df = df.copy()
        df['RSI'] = ta.rsi(df['Close'], length=14)

        macd = ta.macd(df['Close'])
        df = pd.concat([df, macd], axis=1)

        bbands = ta.bbands(df['Close'], length=20)
        df = pd.concat([df, bbands], axis=1)

        df['Log_Ret'] = np.log(df['Close'] / df['Close'].shift(1))

        # Target: 1 nếu giá ngày mai cao hơn hôm nay
        df['Target'] = (df['Close'].shift(-1) > df['Close']).astype(int)

        return df.dropna()

    def create_sliding_windows(self, df: pd.DataFrame, feature_cols: List[str]) -> Tuple[np.ndarray, np.ndarray]:
        """Tạo tensor 3D cho LSTM và nhãn."""
        data = df[feature_cols].values
        labels = df['Target'].values

        X, y = [], []
        for i in range(len(data) - self.window_size):
            X.append(data[i:i + self.window_size])
            y.append(labels[i + self.window_size - 1])

        return np.array(X), np.array(y)

    def time_series_split(self, X: np.ndarray, y: np.ndarray, train_ratio: float = 0.8) -> Tuple:
        """Chia tập dữ liệu theo thời gian để chống rò rỉ dữ liệu."""
        split_idx = int(len(X) * train_ratio)
        X_train, X_test = X[:split_idx], X[split_idx:]
        y_train, y_test = y[:split_idx], y[split_idx:]

        # Flatten để scale sau đó reshape lại
        N, T, F = X_train.shape
        X_train_flat = self.scaler.fit_transform(X_train.reshape(-1, F))
        X_test_flat = self.scaler.transform(X_test.reshape(-1, F))

        return X_train_flat.reshape(N, T, F), X_test_flat.reshape(-1, T, F), y_train, y_test

Overwriting /content/drive/MyDrive/DeTaiHocSauKetHopDoAnTongHop/Stock_Forecasting_Project/04_Source_Code/feature_engineering.py


In [ ]:
import os

# 1. Đảm bảo thư mục tồn tại
path = "/content/drive/MyDrive/DeTaiHocSauKetHopDoAnTongHop/Stock_Forecasting_Project/04_Source_Code/"
os.makedirs(path, exist_ok=True)

# 2. Nội dung file model_architecture.py
# Sử dụng triple single quotes để bao bọc code có chứa triple double quotes
code_content = r'''import tensorflow as tf
from tensorflow.keras import layers, Model

class TimeAttentionLayer(layers.Layer):
    """
    Custom Attention Layer: Tính toán trọng số quan trọng của các bước thời gian.
    Tích hợp get_config để hỗ trợ model.save() .h5
    """
    def __init__(self, **kwargs):
        super(TimeAttentionLayer, self).__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(name="att_weight",
                                 shape=(input_shape[-1], 1),
                                 initializer="normal")
        self.b = self.add_weight(name="att_bias",
                                 shape=(input_shape[1], 1),
                                 initializer="zeros")
        super(TimeAttentionLayer, self).build(input_shape)

    def call(self, x):
        e = tf.tanh(tf.tensordot(x, self.W, axes=1) + self.b)
        a = tf.nn.softmax(e, axis=1)
        output = x * a
        return tf.reduce_sum(output, axis=1)

    def get_config(self):
        config = super(TimeAttentionLayer, self).get_config()
        return config

class DataFusionModel:
    def __init__(self, sequence_length=30, n_features=10, nlp_dim=3):
        self.sequence_length = sequence_length
        self.n_features = n_features
        self.nlp_dim = nlp_dim

    def build_model(self):
        # Nhánh Time-series
        ts_input = layers.Input(shape=(self.sequence_length, self.n_features), name="ts_input")
        x_ts = layers.LSTM(64, return_sequences=True)(ts_input)
        x_ts = layers.Dropout(0.2)(x_ts)
        x_ts = layers.LSTM(32, return_sequences=True)(x_ts)

        # Tích hợp Attention
        ts_attended = TimeAttentionLayer(name="time_attention")(x_ts)

        # Nhánh NLP
        nlp_input = layers.Input(shape=(self.nlp_dim,), name="nlp_input")
        x_nlp = layers.Dense(16, activation='relu')(nlp_input)
        x_nlp = layers.BatchNormalization()(x_nlp)

        # Fusion
        combined = layers.Concatenate()([ts_attended, x_nlp])
        x = layers.Dense(32, activation='relu')(combined)
        x = layers.Dropout(0.3)(x)

        # Multi-step Output (Dự báo 3 ngày tiếp theo)
        output = layers.Dense(3, activation='linear', name="price_prediction")(x)

        model = Model(inputs=[ts_input, nlp_input], outputs=output)
        model.compile(optimizer='adam', loss='huber_loss', metrics=['mae'])
        return model
'''

# 3. Ghi file
file_full_path = os.path.join(path, "model_architecture.py")
with open(file_full_path, "w", encoding="utf-8") as f:
    f.write(code_content.strip())

print(f"Thành công! File đã được lưu tại: {file_full_path}")

Thành công! File đã được lưu tại: /content/drive/MyDrive/DeTaiHocSauKetHopDoAnTongHop/Stock_Forecasting_Project/04_Source_Code/model_architecture.py


In [10]:
%%writefile /content/drive/MyDrive/DeTaiHocSauKetHopDoAnTongHop/Stock_Forecasting_Project/04_Source_Code/backtest_engine.py
import pandas as pd
import numpy as np

class BacktestEngine:
    """
    Hệ thống Backtest tích hợp quản trị rủi ro:
    1. Hard Stop-Loss: Cắt lỗ khi chạm ngưỡng cố định.
    2. Position Sizing: Chỉ giải ngân một phần vốn cho mỗi vị thế.
    """
    def __init__(self, initial_capital=100000000.0, stop_loss_pct=-0.07, position_size_pct=0.2):
        self.initial_capital = initial_capital
        self.stop_loss_pct = stop_loss_pct
        self.position_size_pct = position_size_pct

    def run(self, df, signals):
        """Alias for run_backtest"""
        return self.run_backtest(df, signals)

    def run_backtest(self, df, signals):
        """
        signals: Mảng dự báo từ AI (1: Mua, 0: Không hành động)
        df: DataFrame chứa giá 'Close'
        """
        capital = self.initial_capital
        position = 0
        buy_price = 0
        history = []

        # Giả sử signals khớp với index của df
        for i in range(len(signals)):
            current_price = df.iloc[i]['Close']
            current_signal = signals[i]

            # 1. KIỂM TRA STOP LOSS (Nếu đang giữ hàng)
            if position > 0:
                unrealized_return = (current_price - buy_price) / buy_price
                if unrealized_return <= self.stop_loss_pct:
                    # Ép bán lập tức (Hard Stop-Loss)
                    capital += position * current_price
                    position = 0
                    buy_price = 0
                    print(f"Day {i}: Stop-Loss triggered at {current_price}")

            # 2. LOGIC GIAO DỊCH
            # Mua nếu AI báo Tăng (1) và chưa có vị thế
            if current_signal == 1 and position == 0:
                # Position Sizing: Chỉ dùng X% vốn để mua
                investment = capital * self.position_size_pct
                capital -= investment
                position = investment / current_price
                buy_price = current_price

            # Bán nếu AI báo Giảm (0) và đang có vị thế (Chốt lời/Cắt lỗ theo AI)
            elif current_signal == 0 and position > 0:
                capital += position * current_price
                position = 0
                buy_price = 0

            # Tính tổng tài sản (Vốn mặt + Giá trị cổ phiếu đang giữ)
            total_assets = capital + (position * current_price)
            history.append(total_assets)

        return pd.Series(history, index=df.index)

Overwriting /content/drive/MyDrive/DeTaiHocSauKetHopDoAnTongHop/Stock_Forecasting_Project/04_Source_Code/backtest_engine.py


In [ ]:
%%writefile /content/drive/MyDrive/DeTaiHocSauKetHopDoAnTongHop/Stock_Forecasting_Project/04_Source_Code/data_ingestion.py
import pandas as pd
import numpy as np
import yfinance as yf
from typing import Optional

class DataIngestor:
    """Module chịu trách nhiệm thu thập và làm sạch dữ liệu chứng khoán."""
    def __init__(self, ticker: str, start_date: str, end_date: str):
        self.ticker = ticker
        self.start_date = start_date
        self.end_date = end_date

    def fetch_stock_data(self) -> pd.DataFrame:
        """Lấy dữ liệu OHLCV và xử lý dữ liệu trống."""
        print(f"Fetching data for {self.ticker}...")
        df = yf.download(self.ticker, start=self.start_date, end=self.end_date, auto_adjust=True)

        # Xử lý các ngày nghỉ lễ/cuối tuần bị thiếu bằng Forward Fill
        df = df.reindex(pd.date_range(start=self.start_date, end=self.end_date, freq='B'))
        df = df.ffill()

        # Xử lý MultiIndex nếu yfinance trả về
        if isinstance(df.columns, pd.MultiIndex):
            df.columns = df.columns.get_level_values(0)

        df.index.name = 'time'
        return df.dropna()

Overwriting /content/drive/MyDrive/DeTaiHocSauKetHopDoAnTongHop/Stock_Forecasting_Project/04_Source_Code/data_ingestion.py


In [ ]:
%%writefile /content/drive/MyDrive/DeTaiHocSauKetHopDoAnTongHop/Stock_Forecasting_Project/04_Source_Code/feature_engineering.py
import pandas as pd
import numpy as np
import pandas_ta as ta
from sklearn.preprocessing import StandardScaler
from typing import Tuple, List

class FeatureEngineer:
    def __init__(self, window_size: int = 30, forecast_horizon: int = 3):
        self.window_size = window_size
        self.forecast_horizon = forecast_horizon
        self.scaler = StandardScaler()

    def add_indicators(self, df: pd.DataFrame) -> pd.DataFrame:
        df = df.copy()
        df['RSI'] = ta.rsi(df['Close'], length=14)
        macd = ta.macd(df['Close'])
        df = pd.concat([df, macd], axis=1)
        bbands = ta.bbands(df['Close'], length=20)
        df = pd.concat([df, bbands], axis=1)
        df['Log_Ret'] = np.log(df['Close'] / df['Close'].shift(1))

        # SỬA LỖI: Tạo nhãn cho 3 ngày tiếp theo (Multi-step Target)
        for i in range(1, self.forecast_horizon + 1):
            df[f'Target_{i}'] = (df['Close'].shift(-i) > df['Close']).astype(int)

        return df.dropna()

    def create_sliding_windows(self, df: pd.DataFrame, feature_cols: List[str]) -> Tuple[np.ndarray, np.ndarray]:
        data = df[feature_cols].values
        # Lấy tất cả các cột Target đã tạo
        target_cols = [f'Target_{i}' for i in range(1, self.forecast_horizon + 1)]
        labels = df[target_cols].values

        X, y = [], []
        for i in range(len(data) - self.window_size):
            X.append(data[i:i + self.window_size])
            y.append(labels[i + self.window_size - 1])

        return np.array(X), np.array(y)

    def time_series_split(self, X: np.ndarray, y: np.ndarray, train_ratio: float = 0.8) -> Tuple:
        split_idx = int(len(X) * train_ratio)
        X_train, X_test = X[:split_idx], X[split_idx:]
        y_train, y_test = y[:split_idx], y[split_idx:]

        N, T, F = X_train.shape
        X_train_flat = self.scaler.fit_transform(X_train.reshape(-1, F))
        X_test_flat = self.scaler.transform(X_test.reshape(-1, F))

        return X_train_flat.reshape(N, T, F), X_test_flat.reshape(-1, T, F), y_train, y_test

Overwriting /content/drive/MyDrive/DeTaiHocSauKetHopDoAnTongHop/Stock_Forecasting_Project/04_Source_Code/feature_engineering.py


In [ ]:
import os

# 1. Đường dẫn lưu file
path = "/content/drive/MyDrive/DeTaiHocSauKetHopDoAnTongHop/Stock_Forecasting_Project/04_Source_Code/"
file_full_path = os.path.join(path, "model_architecture.py")

# 2. Nội dung file hỗ trợ Multi-step (3 ngày)
code_content = r'''import tensorflow as tf
from tensorflow.keras import layers, Model

class TimeAttentionLayer(layers.Layer):
    def __init__(self, **kwargs):
        super(TimeAttentionLayer, self).__init__(**kwargs)

    def build(self, input_shape):
        self.W = self.add_weight(name="att_weight",
                                 shape=(input_shape[-1], 1),
                                 initializer="normal")
        self.b = self.add_weight(name="att_bias",
                                 shape=(input_shape[1], 1),
                                 initializer="zeros")
        super(TimeAttentionLayer, self).build(input_shape)

    def call(self, x):
        e = tf.tanh(tf.tensordot(x, self.W, axes=1) + self.b)
        a = tf.nn.softmax(e, axis=1)
        output = x * a
        return tf.reduce_sum(output, axis=1)

    def get_config(self):
        config = super(TimeAttentionLayer, self).get_config()
        return config

class DataFusionModel:
    def __init__(self, sequence_length=30, n_features=3, nlp_dim=3, forecast_horizon=3):
        self.sequence_length = sequence_length
        self.n_features = n_features
        self.nlp_dim = nlp_dim
        self.forecast_horizon = forecast_horizon

    def build_model(self):
        ts_input = layers.Input(shape=(self.sequence_length, self.n_features), name="ts_input")
        x_ts = layers.LSTM(64, return_sequences=True)(ts_input)
        x_ts = layers.Dropout(0.2)(x_ts)
        x_ts = layers.LSTM(32, return_sequences=True)(x_ts)
        ts_attended = TimeAttentionLayer(name="time_attention")(x_ts)

        nlp_input = layers.Input(shape=(self.nlp_dim,), name="nlp_input")
        x_nlp = layers.Dense(16, activation='relu')(nlp_input)
        x_nlp = layers.BatchNormalization()(x_nlp)

        combined = layers.Concatenate()([ts_attended, x_nlp])
        x = layers.Dense(32, activation='relu')(combined)
        x = layers.Dropout(0.3)(x)

        output = layers.Dense(self.forecast_horizon, activation='linear', name="price_prediction")(x)

        model = Model(inputs=[ts_input, nlp_input], outputs=output)
        model.compile(optimizer='adam', loss=tf.keras.losses.Huber(), metrics=['mae'])
        return model
'''

# 3. Ép ghi đè file
with open(file_full_path, "w", encoding="utf-8") as f:
    f.write(code_content.strip())

print("✅ ĐÃ CẬP NHẬT: model_architecture.py hiện đã hỗ trợ forecast_horizon.")

✅ ĐÃ CẬP NHẬT: model_architecture.py hiện đã hỗ trợ forecast_horizon.


In [ ]:
%%writefile /content/drive/MyDrive/DeTaiHocSauKetHopDoAnTongHop/Stock_Forecasting_Project/04_Source_Code/backtest_engine.py
import pandas as pd
import numpy as np

class BacktestEngine:
    """
    Hệ thống Backtest tích hợp quản trị rủi ro:
    1. Hard Stop-Loss: Cắt lỗ khi chạm ngưỡng cố định.
    2. Position Sizing: Chỉ giải ngân một phần vốn cho mỗi vị thế.
    """
    def __init__(self, initial_capital=100000000.0, stop_loss_pct=-0.07, position_size_pct=0.2):
        self.initial_capital = initial_capital
        self.stop_loss_pct = stop_loss_pct
        self.position_size_pct = position_size_pct

    def run(self, df, signals):
        """Alias for run_backtest"""
        return self.run_backtest(df, signals)

    def run_backtest(self, df, signals):
        """
        signals: Mảng dự báo từ AI (1: Mua, 0: Không hành động)
        df: DataFrame chứa giá 'Close'
        """
        capital = self.initial_capital
        position = 0
        buy_price = 0
        history = []

        # Giả sử signals khớp với index của df
        for i in range(len(signals)):
            current_price = df.iloc[i]['Close']
            current_signal = signals[i]

            # 1. KIỂM TRA STOP LOSS (Nếu đang giữ hàng)
            if position > 0:
                unrealized_return = (current_price - buy_price) / buy_price
                if unrealized_return <= self.stop_loss_pct:
                    # Ép bán lập tức (Hard Stop-Loss)
                    capital += position * current_price
                    position = 0
                    buy_price = 0
                    print(f"Day {i}: Stop-Loss triggered at {current_price}")

            # 2. LOGIC GIAO DỊCH
            # Mua nếu AI báo Tăng (1) và chưa có vị thế
            if current_signal == 1 and position == 0:
                # Position Sizing: Chỉ dùng X% vốn để mua
                investment = capital * self.position_size_pct
                capital -= investment
                position = investment / current_price
                buy_price = current_price

            # Bán nếu AI báo Giảm (0) và đang có vị thế (Chốt lời/Cắt lỗ theo AI)
            elif current_signal == 0 and position > 0:
                capital += position * current_price
                position = 0
                buy_price = 0

            # Tính tổng tài sản (Vốn mặt + Giá trị cổ phiếu đang giữ)
            total_assets = capital + (position * current_price)
            history.append(total_assets)

        return pd.Series(history, index=df.index)

Overwriting /content/drive/MyDrive/DeTaiHocSauKetHopDoAnTongHop/Stock_Forecasting_Project/04_Source_Code/backtest_engine.py
